# Task 3: train and evaluate meta-review models on Colab

Runtime → Change runtime type → **T4 GPU** (or L4/A100 if available). Then run the cells in order.

**Before you start:** upload your local GameWikiSum folder (`train.json`, `val.json`, `test.json`)
to Google Drive at `MyDrive/gamewikisum/` (or change `DATA_DRIVE` below).

Everything (models, metrics, predictions) is saved to `MyDrive/steam-review-translator-task3/`,
so a disconnect does not lose finished work. Re-running a cell skips a model or evaluation that
is already saved.

| System | What |
|---|---|
| B1 | extractive TextRank (k=3), no ML |
| B2 | `facebook/bart-large-cnn` zero-shot |
| E1 | BART-base fine-tuned, text only |
| E2 | BART-base fine-tuned + aspect prefix (main system) |
| E3 | T5-base fine-tuned + aspect prefix (optional) |

**Time budget (T4, rough):** each fine-tune at `PILOT=2000` takes about 30-45 min; each test
evaluation at `TEST_LIMIT=500` about 10-20 min. `PILOT=0` = full train split (hours per model).

In [ ]:
# --- settings -------------------------------------------------------------------------
BRANCH = 'task3/local-gamewikisum'   # change to 'main' once that branch is merged
DATA_DRIVE = '/content/drive/MyDrive/gamewikisum'
OUT = '/content/drive/MyDrive/steam-review-translator-task3'
PILOT = 2000        # train documents per model (0 = full train split)
EPOCHS = 3
TEST_LIMIT = 500    # test documents to evaluate (None = all 1,454)
RUN_T5 = False      # E3 is optional; set True if there is time
SEED = 42

from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs(OUT, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Clone the repo and install Task 3 dependencies.
!rm -rf /content/srt
!git clone -q --branch $BRANCH https://github.com/mahesa005/steam-review-translator.git /content/srt
%cd /content/srt
!pip install -q -e ".[task3]" 2>&1 | grep -v -i "warning" | tail -3

In [ ]:
# Copy the data to the local disk (much faster to read than Drive).
!mkdir -p /content/gws && cp -v $DATA_DRIVE/train.json $DATA_DRIVE/val.json $DATA_DRIVE/test.json /content/gws/
!ls -la /content/gws
DATA = '/content/gws'

## 1. Dataset statistics (for the paper and slides)

In [ ]:
import sys, json, statistics
from pathlib import Path
sys.path.insert(0, 'experiments/task3_summarization')
from common import load_local_split

stats = {}
for split in ['train', 'val', 'test']:
    rows = load_local_split(Path(DATA), split)
    para = [len(r['preprocessed_paragraphs_per_reviews']) for r in rows]
    in_words = [sum(len(p.split()) for p in r['preprocessed_paragraphs_per_reviews']) for r in rows]
    sum_words = [len(r['preprocessed_summary'].split()) for r in rows]
    stats[split] = {
        'documents': len(rows),
        'games': len({r['game_title'] for r in rows}),
        'paragraphs_per_doc_mean (cap 400)': round(statistics.mean(para), 1),
        'input_words_mean (cap 400 paragraphs)': round(statistics.mean(in_words)),
        'summary_words_mean': round(statistics.mean(sum_words), 1),
        'summary_words_median': statistics.median(sum_words),
    }
    if split == 'train':
        example = rows[0]
    del rows
print(json.dumps(stats, indent=2))
json.dump(stats, open(f'{OUT}/dataset_stats.json', 'w'), indent=2)

print('\n=== training example ===')
print('game:', example['game_title'])
print('review paragraphs (first 3):')
for p in example['preprocessed_paragraphs_per_reviews'][:3]:
    print('  -', p[:300])
print('target summary:', example['preprocessed_summary'][:800])

In [ ]:
# The same example after preprocessing: the exact model input (aspect prefix + selected text).
from transformers import AutoTokenizer
from common import build_examples, normalize_rows
tok = AutoTokenizer.from_pretrained('facebook/bart-base')
rows = normalize_rows([example], 'preprocessed_paragraphs_per_reviews', 'preprocessed_summary',
                      'game_title')
ex, _ = build_examples(rows, tokenizer=tok, budget_tokens=800, max_target_tokens=256, top_k=5)
if ex:
    print('INPUT:', ex[0].input_text[:1200])
    print('\nTARGET:', ex[0].target[:600])
else:
    print('this document has < 100 words after cleaning; pick another one')

## 2. Fine-tune (each model is saved to Drive)

In [ ]:
def train(script, out_dir, extra=''):
    if os.path.exists(f'{out_dir}/config.json'):
        print('already trained, skipping:', out_dir)
        return
    cmd = (f'python experiments/task3_summarization/{script} --dataset {DATA} --output {out_dir} '
           f'--pilot-samples {PILOT} --epochs {EPOCHS} --seed {SEED} --fp16 {extra}')
    print(cmd)
    get_ipython().system(cmd + ' 2>&1 | grep -E "eval_rouge|Error|error|disimpan"')

# E2 (main system): BART-base + aspect prefix
train('train_bart.py', f'{OUT}/models/e2_bart', '--aspect-prefix')

In [ ]:
# E1: BART-base, text only (shows whether the aspect prefix helps)
train('train_bart.py', f'{OUT}/models/e1_bart')

In [ ]:
# E3 (optional): T5-base + aspect prefix
if RUN_T5:
    train('train_t5.py', f'{OUT}/models/e3_t5')

## 3. Evaluate on test (each system once)

In [ ]:
LIMIT = f'--limit {TEST_LIMIT}' if TEST_LIMIT else ''
R = f'{OUT}/runs'
os.makedirs(R, exist_ok=True)

def evaluate(name, model, extra=''):
    if os.path.exists(f'{R}/{name}.json'):
        print('already evaluated, skipping:', name)
        return
    cmd = (f'python experiments/task3_summarization/evaluate.py --dataset {DATA} --split test '
           f'--model {model} --name {name} {LIMIT} --out {R}/{name}.json '
           f'--records {R}/{name}_records.json {extra}')
    print(cmd)
    get_ipython().system(cmd + ' 2>&1 | grep -v -i warning | tail -25')

evaluate('B1', 'facebook/bart-base', '--extractive textrank --aspect-prefix')
evaluate('B2', 'facebook/bart-large-cnn')
evaluate('E1', f'{OUT}/models/e1_bart')
evaluate('E2', f'{OUT}/models/e2_bart', f'--aspect-prefix --baseline {R}/E1.json')   # E2 vs E1
if RUN_T5:
    evaluate('E3', f'{OUT}/models/e3_t5', f'--aspect-prefix --baseline {R}/E2.json')  # E3 vs E2

## 4. Results table

In [ ]:
import glob
table = []
for f in sorted(glob.glob(f'{R}/*.json')):
    if f.endswith('_records.json') or f.endswith('error_analysis.json'):
        continue
    r = json.load(open(f))
    g = r['rouge']
    table.append((r['system'], r['n_samples'], g['rouge1'], g['rouge2'], g['rougeL'],
                  r['aspect_coverage'], r['polarity_consistency'],
                  r.get('bootstrap_vs_baseline', '')))
print(f"{'system':<6} {'n':>5} {'R-1':>6} {'R-2':>6} {'R-L':>6} {'asp.cov':>8} {'pol.cons':>9}")
for s, n, r1, r2, rl, cov, pol, bs in table:
    print(f'{s:<6} {n:>5} {r1:>6.3f} {r2:>6.3f} {rl:>6.3f} {cov:>8.3f} {pol:>9.3f}')
for s, *_, bs in table:
    if bs:
        print(f'{s} paired bootstrap vs baseline:', bs)

## 5. Error analysis

In [ ]:
systems = ' '.join(f'--system {s}={R}/{s}_records.json' for s in ['B2', 'E1', 'E2', 'E3']
                   if os.path.exists(f'{R}/{s}_records.json'))
get_ipython().system(f'python experiments/task3_summarization/error_analysis.py {systems} '
                     f'--out {R}/error_analysis.json 2>&1 | grep -v -i warning')

In [ ]:
# Example outputs for the slides: worst, median and best E2 sample by ROUGE-L.
recs = json.load(open(f'{R}/E2_records.json'))
order = sorted(range(len(recs)), key=lambda i: recs[i]['rougeL'])
for label, i in [('worst', order[0]), ('median', order[len(order) // 2]), ('best', order[-1])]:
    r = recs[i]
    print(f"=== {label}: {r['game']} (ROUGE-L {r['rougeL']:.3f}) ===")
    print('aspects  :', r['aspects'])
    print('predicted:', r['prediction'][:700])
    print('reference:', r['reference'][:700], '\n')

## 6. Send back

Copy the printed results table, the error-analysis table and the example outputs into
`experiments/task3_summarization/RESULTS.md` (or the group chat). Everything is also in
`MyDrive/steam-review-translator-task3/` (`dataset_stats.json`, `runs/*.json`, `models/`).
Do **not** commit model weights or GameWikiSum data to git.